Data Analyst: Andrei Berner Arroyo Tanta

**Project with Numpy 3: Automated Financial Solvency & Liquidity Engine

**Business Application: This project addresses the need for automated assessment of corporate financial health across thousands of records. By transforming raw balance sheet data into weighted liquidity metrics, it allows Treasury and Credit Risk departments to move beyond simple numbers to instant, actionable insights. This system improves decision-making by identifying which companies possess high-quality liquid assets to meet short-term obligations.

**Executive Summary

Key Findings: Processed 1,781 financial records efficiently. The analysis identified SWKS (2016) as highly solvent with a ratio of 5.56, while flagging companies like KMX and HCA with critical liquidity risks.

Risks Detected: Automated risk categorization labels firms as "At Risk (Underfunded)" if they fall below a 1.0 threshold, enabling immediate audit prioritization.

In [21]:
#Environment Setup & Data Ingestion
import numpy as np
import pandas as pd

In [22]:
#Load dataset
df_raw = pd.read_csv("fundamentals.csv")

#Column Selection & Memory Optimization
target_columns = [
    "Ticker Symbol", "Period Ending", "Total Current Assets", 
    "Total Current Liabilities", "Cash and Cash Equivalents", 
    "Accounts Receivable", "Accounts Payable", "Total Revenue"
]

df_work = df_raw.reindex(columns=target_columns).copy()

#Data Type Transformation
df_work["Period Ending"] = pd.to_datetime(df_work["Period Ending"])

#Vectorized conversion of financial metrics to float32
financial_cols = df_work.columns[2:]
df_work[financial_cols] = df_work[financial_cols].apply(pd.to_numeric, errors='coerce').astype('float32')

print(f"Data ingested: {df_work.shape[0]} financial records optimized to float32.")

Data ingested: 1781 financial records optimized to float32.


In [28]:
#Weighted Liquidity Engine (Numpy Layer)

#Define Liquidity Weights
liquidity_weights = np.array([1.0, 0.7], dtype='float32')

#Matrix Multiplication
#We extract the relevant columns as a NumPy matrix and multiply by the weights vector
liquidity_matrix = df_work[["Cash and Cash Equivalents", "Accounts Receivable"]].values
df_work["Weighted_Liquidity"] = np.dot(liquidity_matrix, liquidity_weights)

In [24]:
#Custom Analytical Logic (Pandas Layer)
def calculate_solvency_ratio(row):
    """Calculates the ratio of weighted assets vs short-term debt."""
    liabilities = row['Total Current Liabilities']
    weighted_assets = row['Weighted_Liquidity']
    
    return weighted_assets / liabilities if liabilities > 0 else 0.0

# Applying custom logic across the DataFrame
df_work['Solvencia_Ratio'] = df_work.apply(calculate_solvency_ratio, axis=1)

In [25]:
#Risk Categorization
# Categorization based on the Quick Ratio threshold (1.0)
df_work["Risk_Category"] = np.where(df_work["Solvencia_Ratio"] < 1.0, 
    "At Risk (Underfunded)", "Healthy (Liquid)")

In [34]:
# Aggregation & Executive Reporting
# Extracting Year for time-series grouping
df_work["Year"] = df_work["Period Ending"].dt.year

# Annual Solvency Summary
solvency_summary = df_work.groupby(["Ticker Symbol", "Year"]).agg({
    "Solvencia_Ratio": "mean",
    "Total Current Liabilities": "sum",
    "Risk_Category": "first"
}).reset_index()

print(f"--- Report ---")
# Ranking: The Top 5 Most Solvent Companies vs. Bottom 5
print(f"--- Top 5 Most Solvent Records ---")
display(solvency_summary.sort_values(by="Solvencia_Ratio", ascending=False).head(5))

print(f"--- Bottom 5 Records (Highest Liquidity Risk) ---")
display(solvency_summary.sort_values(by="Solvencia_Ratio", ascending=True).head(5))

--- Report ---
--- Top 5 Most Solvent Records ---


,Ticker Symbol,Year,Solvencia_Ratio,Total Current Liabilities,Risk_Category
1469,SWKS,2016,5.560324,2.102000e+08,Healthy (Liquid)
1036,MCHP,2016,5.474337,3.820090e+08,Healthy (Liquid)
678,FTR,2015,4.978024,1.893000e+09,Healthy (Liquid)
1080,MNST,2015,4.126314,5.141890e+08,Healthy (Liquid)
922,KORS,2014,2.912456,3.083700e+08,Healthy (Liquid)


--- Bottom 5 Records (Highest Liquidity Risk) ---


,Ticker Symbol,Year,Solvencia_Ratio,Total Current Liabilities,Risk_Category
915,KMX,2015,-0.679630,9.971730e+08,At Risk (Underfunded)
747,HCA,2013,-0.467515,5.695000e+09,At Risk (Underfunded)
916,KMX,2016,-0.454365,1.005193e+09,At Risk (Underfunded)
749,HCA,2015,-0.387745,5.516000e+09,At Risk (Underfunded)
748,HCA,2014,-0.362318,5.480000e+09,At Risk (Underfunded)
